# RAG Assignment: Baseline Retrieval vs Cross-Encoder Re-Ranking
**Document:** FIDE Laws of Chess (25 pages, `doc/laws_of_chess.pdf`). It is full of exceptions, penalties and "not allowed" clauses, which makes it a good stress test.

**Pipeline 1:** bi-encoder top-10 -> top-3 -> LLM (Answer 1)
**Pipeline 2:** same top-10 -> cross-encoder re-rank -> top-3 -> LLM (Answer 2)

Both pipelines share the same chunks, the same candidate set and the exact same prompt.

## 1. Setup

In [1]:
# pip install sentence-transformers faiss-cpu pypdf pandas torch google-genai
# The Gemini key is read from the GEMINI_API_KEY environment variable (never hard-code it).
import os
import re, time, textwrap
import numpy as np, pandas as pd, faiss, torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

PDF_PATH   = "doc/laws_of_chess.pdf"
BI_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"          # bi-encoder
CE_MODEL   = "cross-encoder/ms-marco-MiniLM-L-6-v2"            # cross-encoder
LLM_MODEL  = "gemini-3.1-flash-lite-preview"                         # Gemini API
CHUNK_WORDS, STRIDE = 100, 80                                   # 20-word overlap
TOP_K, TOP_N = 10, 3
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)

/Users/rick/code/IIIT/SEM3/lma/AS1/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1019 12:54:36.352000 65792 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 2. Load PDF and chunk it
Fixed-size word windows with a small overlap, remembering the page number of each chunk. Both pipelines use this identical corpus.

In [2]:
words = []
for pno, page in enumerate(PdfReader(PDF_PATH).pages, 1):
    for w in re.sub(r"\s+", " ", page.extract_text() or "").split(" "):
        if w: words.append((w, pno))

chunks = []   # list of dicts: id, page, text
for i in range(0, len(words), STRIDE):
    seg = words[i:i + CHUNK_WORDS]
    chunks.append({"id": len(chunks), "page": seg[0][1], "text": " ".join(w for w, _ in seg)})
    if i + CHUNK_WORDS >= len(words): break
print(f"{len(words)} words -> {len(chunks)} chunks (about {CHUNK_WORDS} words each)")

9662 words -> 121 chunks (about 100 words each)


## 3. Pipeline 1: dense index (Bi-Encoder + FAISS inner product)
Embeddings are L2-normalised, so inner product equals cosine similarity.

In [3]:
bi_encoder = SentenceTransformer(BI_MODEL)
doc_emb = bi_encoder.encode([c["text"] for c in chunks], normalize_embeddings=True, show_progress_bar=False)
index = faiss.IndexFlatIP(doc_emb.shape[1]); index.add(doc_emb)

def retrieve_baseline(query, k=TOP_K):
    q = bi_encoder.encode([query], normalize_embeddings=True)
    scores, ids = index.search(q, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]     # ranked best first

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11964.15it/s]

## 4. Pipeline 2: Cross-Encoder re-ranking
Takes the *same* 10 candidates, scores every (query, chunk) pair jointly and re-sorts.

In [4]:
cross_encoder = CrossEncoder(CE_MODEL)

def rerank(query, candidates):
    pairs = [(query, chunks[i]["text"]) for i, _ in candidates]
    ce_scores = cross_encoder.predict(pairs, show_progress_bar=False)
    order = np.argsort(-ce_scores)
    return [(candidates[j][0], float(ce_scores[j])) for j in order]    # ranked best first

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7978.58it/s]

## 5. LLM generation (identical prompt for both answers)

In [5]:
from google import genai
from google.genai import types
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

PROMPT = '''You are answering questions about the FIDE Laws of Chess.
Use ONLY the context below. If the context does not contain the answer, say "Not found in the provided context."
Be concise (max 3 sentences).

Context:
{context}

Question: {question}
Answer:'''

import json, hashlib
CACHE_FILE = "llm_cache.json"          # free-tier quota is small, so answers are cached between runs
cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}

def generate(question, chunk_ids, use_cache=True):
    context = "\n\n".join(f"[Chunk {i}, p.{chunks[i]['page']}] {chunks[i]['text']}" for i in chunk_ids)
    prompt = PROMPT.format(context=context, question=question)
    key = hashlib.md5((LLM_MODEL + prompt).encode()).hexdigest()
    if use_cache and key in cache: return cache[key]
    for attempt in range(3):                      # retry transient API errors
        try:
            r = client.models.generate_content(model=LLM_MODEL, contents=prompt,
                    config=types.GenerateContentConfig(temperature=0, max_output_tokens=2000))
            cache[key] = (r.text or "").strip()
            json.dump(cache, open(CACHE_FILE, "w"))
            return cache[key]
        except Exception as e:
            err = e
            if "429" in str(e): break             # quota exhausted, retrying will not help
            time.sleep(2 * (attempt + 1))
    return f"[LLM error: {str(err)[:150]}]"

## 6. Stress-test queries
For each query I mark the *gold* chunk(s) by a phrase that must appear in the clause that really answers it (found by reading the rulebook, not by running the models). The notebook then reports where the gold chunk ranks under the bi-encoder and under the cross-encoder.

In [6]:
TESTS = [
 # (category, query, gold phrase present in the answering clause)
 ("Failure query 1 (gold chunk #4 -> #1)", "How many extra minutes does the opponent get when the arbiter rejects a draw claim?", "rejected the claim, the opponent shall be awarded two extra minutes"),
 ("Control (easy query, no shift expected)", "What happens if a player makes a third illegal move?", "third illegal move by the same player"),
 ("Keyword trap / synonyms (re-ranker does not help)",               "How long can a player be absent before the game is forfeited?", "default time"),
 ("Failure query 2 + Exception / corner case",            "If my opponent's time runs out, do I always win?", "cannot win the game by any series of legal moves"),
 ("Negation / constraint",                 "Is a player allowed to withdraw a draw claim?", "not allowed to withdraw his claim"),
 ("Negation / constraint",                 "Is it prohibited to write the moves in advance on the scoresheet?", "write the moves in advance"),
 ("Other (already top-ranked)",                "What are the rules about mobile phones during play?", "mobile phone"),
]

norm = lambda s: re.sub(r"\s+", " ", s).lower()
def gold_ids(phrase):
    return {c["id"] for c in chunks if norm(phrase) in norm(c["text"])}
for cat, q, g in TESTS:
    assert gold_ids(g), f"gold phrase not found: {g}"
print("all gold phrases located in the document")

all gold phrases located in the document


In [7]:
def best_rank(ranking, gold):
    for r, (cid, _) in enumerate(ranking, 1):
        if cid in gold: return r
    return None     # gold chunk not even in the top 10

results = []
for cat, q, g in TESTS:
    gold = gold_ids(g)
    base = retrieve_baseline(q)
    rer  = rerank(q, base)
    a1 = generate(q, [i for i, _ in base[:TOP_N]])
    a2 = generate(q, [i for i, _ in rer[:TOP_N]])
    br, rr = best_rank(base, gold), best_rank(rer, gold)
    results.append(dict(cat=cat, q=q, gold=sorted(gold), base=base, rer=rer, bi_rank=br, ce_rank=rr, a1=a1, a2=a2))
    print("="*100); print(f"[{cat}] {q}")
    print(f"gold chunk(s) {sorted(gold)}:  bi-encoder rank #{br}  ->  cross-encoder rank #{rr}")
    print("bi top-3 :", [i for i, _ in base[:3]], "| CE top-3:", [i for i, _ in rer[:3]])
    print("\nAnswer 1 (baseline):\n" + textwrap.fill(a1, 110))
    print("\nAnswer 2 (re-ranked):\n" + textwrap.fill(a2, 110))

[Failure query 1 (gold chunk #4 -> #1)] How many extra minutes does the opponent get when the arbiter rejects a draw claim?
gold chunk(s) [63]:  bi-encoder rank #4  ->  cross-encoder rank #1
bi top-3 : [62, 61, 59] | CE top-3: [63, 62, 61]

Answer 1 (baseline):
The opponent shall be awarded two extra minutes.

Answer 2 (re-ranked):
The opponent shall be awarded two extra minutes.
[Control (easy query, no shift expected)] What happens if a player makes a third illegal move?
gold chunk(s) [45]:  bi-encoder rank #1  ->  cross-encoder rank #1
bi top-3 : [45, 25, 44] | CE top-3: [45, 22, 75]

Answer 1 (baseline):
For a third illegal move by the same player, the arbiter shall declare the game lost by that player. However,
the game is drawn if the position is such that the opponent cannot checkmate the player’s king by any possible
series of legal moves.

Answer 2 (re-ranked):
For a third illegal move by the same player, the arbiter shall declare the game lost by that player. However,
the gam

[Negation / constraint] Is a player allowed to withdraw a draw claim?
gold chunk(s) [59]:  bi-encoder rank #1  ->  cross-encoder rank #1
bi top-3 : [59, 55, 53] | CE top-3: [59, 54, 53]

Answer 1 (baseline):
No, a player is not allowed to withdraw their claim.

Answer 2 (re-ranked):
No, a player is not allowed to withdraw a draw claim.
[Negation / constraint] Is it prohibited to write the moves in advance on the scoresheet?
gold chunk(s) [47]:  bi-encoder rank #1  ->  cross-encoder rank #1
bi top-3 : [47, 48, 49] | CE top-3: [47, 48, 52]

Answer 1 (baseline):
Yes, it is forbidden to write the moves in advance, unless the player is claiming a draw according to Article
9.2 or 9.3, or adjourning a game according to the Guidelines of Adjourned Games.

Answer 2 (re-ranked):
Yes, it is forbidden to write the moves in advance, unless the player is claiming a draw according to Article
9.2 or 9.3, or adjourning a game according to the Guidelines of Adjourned Games point 1.a.


[Other (already top-ranked)] What are the rules about mobile phones during play?
gold chunk(s) [65, 72]:  bi-encoder rank #1  ->  cross-encoder rank #1
bi top-3 : [65, 72, 66] | CE top-3: [65, 72, 2]

Answer 1 (baseline):
Players are forbidden from having mobile phones or other electronic communication devices in the playing venue
without the arbiter's permission, unless they are completely switched off. If such a device produces a sound,
the player loses the game, unless the opponent cannot win by any series of legal moves, in which case the
score is a draw. Additionally, it is forbidden for anybody to use such devices in the playing venue or
designated contiguous areas without authorization.

Answer 2 (re-ranked):
Players are forbidden from having mobile phones or other electronic communication devices in the playing venue
without the arbiter's permission, unless they are completely switched off. If such a device produces a sound,
the player loses the game. Additionally, it is forbid

### Rank-shift summary

In [8]:
summary = pd.DataFrame([{ "category": r["cat"], "query": r["q"], "gold chunk(s)": r["gold"],
    "bi-encoder rank": r["bi_rank"], "cross-encoder rank": r["ce_rank"],
    "in top-3 before": (r["bi_rank"] or 99) <= 3, "in top-3 after": (r["ce_rank"] or 99) <= 3} for r in results])
summary

,category,query,gold chunk(s),bi-encoder rank,cross-encoder rank,in top-3 before,in top-3 after
0,Failure query 1 (gold chunk #4 -> #1),How many extra minutes does the opponent get when the arbiter rejects a draw claim?,[63],4,1,False,True
1,"Control (easy query, no shift expected)",What happens if a player makes a third illegal move?,[45],1,1,True,True
2,Keyword trap / synonyms (re-ranker does not help),How long can a player be absent before the game is forfeited?,"[32, 33]",5,9,False,False
3,Failure query 2 + Exception / corner case,"If my opponent's time runs out, do I always win?","[65, 66]",10,1,False,True
4,Negation / constraint,Is a player allowed to withdraw a draw claim?,[59],1,1,True,True
5,Negation / constraint,Is it prohibited to write the moves in advance on the scoresheet?,[47],1,1,True,True
6,Other (already top-ranked),What are the rules about mobile phones during play?,"[65, 72]",1,1,True,True


### Full candidate re-ordering for each query (chunk id: bi rank -> CE rank)

In [9]:
for r in results:
    bi_pos = {cid: k for k, (cid, _) in enumerate(r["base"], 1)}
    row = [f"chunk {cid}{'*' if cid in r['gold'] else ''}: #{bi_pos[cid]}->#{k}" for k, (cid, _) in enumerate(r["rer"], 1)]
    print(r["q"]); print("   " + " | ".join(row)); print("   (* = contains the gold clause)\n")

How many extra minutes does the opponent get when the arbiter rejects a draw claim?
   chunk 63*: #4->#1 | chunk 62: #1->#2 | chunk 61: #2->#3 | chunk 59: #3->#4 | chunk 113: #8->#5 | chunk 66: #9->#6 | chunk 53: #6->#7 | chunk 36: #7->#8 | chunk 120: #10->#9 | chunk 52: #5->#10
   (* = contains the gold clause)

What happens if a player makes a third illegal move?
   chunk 45*: #1->#1 | chunk 22: #6->#2 | chunk 75: #4->#3 | chunk 25: #2->#4 | chunk 58: #10->#5 | chunk 44: #3->#6 | chunk 78: #5->#7 | chunk 26: #7->#8 | chunk 27: #8->#9 | chunk 52: #9->#10
   (* = contains the gold clause)

How long can a player be absent before the game is forfeited?
   chunk 117: #1->#1 | chunk 63: #2->#2 | chunk 118: #3->#3 | chunk 114: #10->#4 | chunk 62: #8->#5 | chunk 116: #6->#6 | chunk 110: #7->#7 | chunk 119: #9->#8 | chunk 33*: #5->#9 | chunk 52: #4->#10
   (* = contains the gold clause)

If my opponent's time runs out, do I always win?
   chunk 66*: #10->#1 | chunk 63: #2->#2 | chunk 33: #1->

## 7. Latency profiling
Retrieval and re-ranking are averaged over 10 repeats per query. LLM generation is timed with one uncached call, because it is identical in both pipelines and the API quota is limited.

In [10]:
def timeit(fn, reps=10):
    ts = []
    for _ in range(reps):
        t = time.perf_counter(); fn(); ts.append((time.perf_counter() - t) * 1000)
    return np.mean(ts), np.std(ts)

queries = [q for _, q, _ in TESTS]
_ = [retrieve_baseline(q) for q in queries[:2]]; _ = rerank(queries[0], retrieve_baseline(queries[0]))   # warm-up

t_base = [timeit(lambda q=q: retrieve_baseline(q))[0] for q in queries]
cands  = [retrieve_baseline(q) for q in queries]
t_rr   = [timeit(lambda q=q, c=c: rerank(q, c))[0] for q, c in zip(queries, cands)]
t_llm  = [np.nan] * len(queries)
t = time.perf_counter(); _ = generate(queries[0], [i for i, _ in cands[0][:3]], use_cache=False); t_llm[0] = (time.perf_counter() - t) * 1000   # one uncached call (API quota is limited)

lat = pd.DataFrame({"query": [q[:60] for q in queries], "Pipeline 1: bi-encoder retrieval (ms)": t_base,
                    "Re-ranking step: 10 CE pairs (ms)": t_rr, "LLM generation (ms)": t_llm})
lat["Pipeline 2 retrieval total (ms)"] = lat.iloc[:, 1] + lat.iloc[:, 2]
display(lat.round(1))
print("MEAN  baseline retrieval : %.1f ms" % np.mean(t_base))
print("MEAN  re-ranking step    : %.1f ms" % np.mean(t_rr))
print("MEAN  pipeline 2 retrieval (bi + CE): %.1f ms  (%.1fx baseline)" % (np.mean(t_base)+np.mean(t_rr), (np.mean(t_base)+np.mean(t_rr))/np.mean(t_base)))
print("LLM generation (single timed call, identical for both pipelines): %.0f ms" % t_llm[0])

# scaling illustration: cross-encoder cost over the *whole* corpus
all_pairs = [(queries[0], c["text"]) for c in chunks]
t = time.perf_counter(); cross_encoder.predict(all_pairs, show_progress_bar=False); full = (time.perf_counter()-t)*1000
print(f"\nCross-encoder over ALL {len(chunks)} chunks for one query: {full:.0f} ms  (vs {np.mean(t_rr):.0f} ms for 10)")

,query,Pipeline 1: bi-encoder retrieval (ms),Re-ranking step: 10 CE pairs (ms),LLM generation (ms),Pipeline 2 retrieval total (ms)
0,How many extra minutes does the opponent get when the arbite,5.1,28.9,1117.0,34.1
1,What happens if a player makes a third illegal move?,4.9,25.8,NaN,30.6
2,How long can a player be absent before the game is forfeited,5.0,27.1,NaN,32.1
3,"If my opponent's time runs out, do I always win?",5.0,30.1,NaN,35.1
4,Is a player allowed to withdraw a draw claim?,4.9,26.6,NaN,31.5
5,Is it prohibited to write the moves in advance on the scores,5.0,27.5,NaN,32.5
6,What are the rules about mobile phones during play?,4.8,25.8,NaN,30.6


MEAN  baseline retrieval : 5.0 ms
MEAN  re-ranking step    : 27.4 ms
MEAN  pipeline 2 retrieval (bi + CE): 32.4 ms  (6.5x baseline)
LLM generation (single timed call, identical for both pipelines): 1117 ms



Cross-encoder over ALL 121 chunks for one query: 398 ms  (vs 27 ms for 10)


## 8. Report notes
The written report (architecture comparison, qualitative analysis, latency table, scaling answer) is in `report.md`; the numbers in it come from the cells above.